# A MNIST model from scratch (using only pytorch)

In [2]:
from datasets import load_dataset
ds = load_dataset("ylecun/mnist")

README.md: 0.00B [00:00, ?B/s]

mnist/train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

mnist/test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [195]:
from torch import nn
import torch
import random
import numpy as np

In [196]:
model = nn.Sequential(
    nn.Linear(28*28, 50),
    nn.ReLU(),
    nn.Linear(50, 30),
    nn.ReLU(),
    nn.Linear(30, 10)
)

In [238]:
def train(model, batch_size, learning_rate, training_dataset, metric, loss, epochs=10):
    for i in range(epochs):
        random.shuffle(training_dataset)
        
        for start_idx in range(0, len(training_dataset), batch_size):
            
            # Extract the current batch
            batch = training_dataset[start_idx : start_idx + batch_size]
            
            x = torch.stack([a for a,_ in batch]).float() / 255.0
            
            y = model(x)
            target = torch.tensor([int(b) for _, b in batch]).long()
            
            l = loss(y, target)
            l.backward()
            
            with torch.no_grad():
                for p in model.parameters():
                    p.data -= p.grad.data * learning_rate
            model.zero_grad()
            
        # Print metrics ONCE at the end of the full epoch
        print("epoch: ", i, "loss: ", l.data.item(), "accuracy: ", metric().item())

In [239]:
def metric():
    x_test = torch.stack([torch.tensor(np.asarray(x['image'])).view(28*28) for x in ds['test']]).float() / 255.0
    y = model(x_test)
    labels = torch.tensor([x['label'] for x in ds['test']])
    return torch.where(y.argmax(-1) == labels, 1, 0).sum(0)/len(ds['test'])

In [240]:
training_dataset = [(torch.tensor(np.asarray(x['image'])).view(28*28), x['label']) for x in ds['train']]

In [242]:
train(model, 256, 0.01, training_dataset, metric, nn.CrossEntropyLoss(), 100)

epoch:  0 loss:  2.2999844551086426 accuracy:  0.17569999396800995
epoch:  1 loss:  2.2216131687164307 accuracy:  0.2565000057220459
epoch:  2 loss:  2.0076169967651367 accuracy:  0.36079999804496765
epoch:  3 loss:  1.5849350690841675 accuracy:  0.4408000111579895
epoch:  4 loss:  1.3399842977523804 accuracy:  0.6672999858856201
epoch:  5 loss:  0.8804268836975098 accuracy:  0.7299000024795532
epoch:  6 loss:  0.839648425579071 accuracy:  0.7646999955177307
epoch:  7 loss:  0.7862858176231384 accuracy:  0.7882999777793884
epoch:  8 loss:  0.7150114178657532 accuracy:  0.8065000176429749
epoch:  9 loss:  0.585077702999115 accuracy:  0.8208000063896179
epoch:  10 loss:  0.5175760984420776 accuracy:  0.8342999815940857
epoch:  11 loss:  0.5005103945732117 accuracy:  0.8422999978065491
epoch:  12 loss:  0.48847225308418274 accuracy:  0.8504999876022339
epoch:  13 loss:  0.5971828103065491 accuracy:  0.8557000160217285
epoch:  14 loss:  0.617060661315918 accuracy:  0.8615999817848206
epoch